# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ssarimm/Machine-Learning-Internship---Flyrank/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
import pandas as pd
from pathlib import Path

# Load the dataset from the current notebook's working directory.
# If the CSV isn't there, use the download fallback below.
csv_path = Path("content_refresh_anonymized.csv")

if not csv_path.exists():
    import urllib.request

    url = (
        "https://raw.githubusercontent.com/"
        "ssarimm/Machine-Learning-Internship---Flyrank/"
        "main/data/raw/content_refresh_anonymized.csv"
    )
    urllib.request.urlretrieve(url, csv_path)

df = pd.read_csv(csv_path)

print("Dataset shape:", df.shape)
print("Required columns available:")
required = [
    "trend_direction",
    "days_since_last_update",
    "search_volume",
    "impressions_90d",
]
print(df[required].isna().sum())
display(df[required].head())

Dataset shape: (30000, 44)
Required columns available:
trend_direction              0
days_since_last_update       0
search_volume             2468
impressions_90d              0
dtype: int64


,trend_direction,days_since_last_update,search_volume,impressions_90d
0,down,20,10.0,3803
1,down,25,90.0,15320
2,down,20,0.0,12581
3,stable,22,10.0,11751
4,down,14,0.0,19140


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

### My baseline rule

My lane is content refresh prioritization. I want to help a content or SEO team decide which pages deserve human review first when time is limited.

I will use a simple score based on two signals:

- **Staleness:** pages not updated for 180 days or more receive 2 points.
- **Search demand:** pages with positive search volume at or above the median positive search volume receive 1 point.

The score ranges from 0 to 3. Pages with higher scores are ranked first. Missing search volume is treated as unknown, not as zero.

Each page receives one reason code:
- `STALE_AND_HIGH_DEMAND`: both signals are present.
- `STALE_ONLY`: the page is stale but does not meet the high-demand condition.
- `HIGH_DEMAND_ONLY`: the page meets the high-demand condition but is not stale.
- `NO_PRIORITY_SIGNAL`: neither signal is present.

The action label is `PRIORITY_REVIEW` for scores of 2 or more, `SECONDARY_REVIEW` for a score of 1, and `NO_PRIORITY_FLAG` for a score of 0.

This is a transparent baseline for decision support, not proof that refreshing a page will recover traffic. The score uses neither the trend label nor performance metrics that might overlap with the target period. Its usefulness must be checked against the observed data and compared with a Week-5 model.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from pathlib import Path
import numpy as np
import pandas as pd

# Check that the required columns exist
required_columns = [
    "content_id",
    "days_since_last_update",
    "search_volume",
]
missing_columns = set(required_columns) - set(df.columns)
assert not missing_columns, f"Missing columns: {missing_columns}"

queue = df[required_columns].copy()

# Signal 1: stale content (180+ days since last update)
queue["is_stale"] = queue["days_since_last_update"] >= 180

# Signal 2: relatively high search demand.
# Calculate the threshold from positive, non-missing values only.
positive_volume = queue.loc[
    queue["search_volume"] > 0, "search_volume"
]
assert not positive_volume.empty, "No positive search-volume values found."

high_demand_threshold = positive_volume.median()

# Missing and zero volume do not qualify as high demand.
queue["is_high_demand"] = (
    queue["search_volume"].notna()
    & (queue["search_volume"] > 0)
    & (queue["search_volume"] >= high_demand_threshold)
)

# Transparent rule: staleness is worth 2 points; high demand is worth 1.
queue["score"] = (
    queue["is_stale"].astype(int) * 2
    + queue["is_high_demand"].astype(int)
)

# Exactly one reason code per page
queue["reason_code"] = np.select(
    [
        queue["is_stale"] & queue["is_high_demand"],
        queue["is_stale"],
        queue["is_high_demand"],
    ],
    [
        "STALE_AND_HIGH_DEMAND",
        "STALE_ONLY",
        "HIGH_DEMAND_ONLY",
    ],
    default="NO_PRIORITY_SIGNAL",
)

# One action label per page
queue["action_label"] = np.select(
    [
        queue["score"] >= 2,
        queue["score"] == 1,
    ],
    [
        "PRIORITY_REVIEW",
        "SECONDARY_REVIEW",
    ],
    default="NO_PRIORITY_FLAG",
)

# Rank by score, then staleness, then search volume.
# Use content_id only to make ties deterministic, not as a feature.
queue = queue.sort_values(
    by=["score", "days_since_last_update", "search_volume", "content_id"],
    ascending=[False, False, False, True],
    na_position="last",
).reset_index(drop=True)

queue.insert(0, "rank", range(1, len(queue) + 1))

# Remove internal helper columns from the deliverable
queue = queue[
    [
        "rank",
        "content_id",
        "score",
        "reason_code",
        "action_label",
        "days_since_last_update",
        "search_volume",
    ]
]

output_path = Path("work/outputs/baseline_action_score.csv")
output_path.parent.mkdir(parents=True, exist_ok=True)
queue.to_csv(output_path, index=False)

print("Positive-volume median threshold:", high_demand_threshold)
print("Pages ranked:", len(queue))
print("Saved CSV to:", output_path)
print("\nAction counts:")
print(queue["action_label"].value_counts(dropna=False))
print("\nReason-code counts:")
print(queue["reason_code"].value_counts(dropna=False))
display(queue.head(10))


Positive-volume median threshold: 20.0
Pages ranked: 30000
Saved CSV to: work/outputs/baseline_action_score.csv

Action counts:
action_label
NO_PRIORITY_FLAG    20693
SECONDARY_REVIEW     9133
PRIORITY_REVIEW       174
Name: count, dtype: int64

Reason-code counts:
reason_code
NO_PRIORITY_SIGNAL       20693
HIGH_DEMAND_ONLY          9133
STALE_ONLY                 167
STALE_AND_HIGH_DEMAND        7
Name: count, dtype: int64


,rank,content_id,score,reason_code,action_label,days_since_last_update,search_volume
0,1,content_02b0d6e30129,3,STALE_AND_HIGH_DEMAND,PRIORITY_REVIEW,313,110.0
1,2,content_7a888d3d99c8,3,STALE_AND_HIGH_DEMAND,PRIORITY_REVIEW,313,90.0
2,3,content_dd413158df3c,3,STALE_AND_HIGH_DEMAND,PRIORITY_REVIEW,305,20.0
3,4,content_bbca724138f2,3,STALE_AND_HIGH_DEMAND,PRIORITY_REVIEW,236,1600.0
4,5,content_40e140ba2934,3,STALE_AND_HIGH_DEMAND,PRIORITY_REVIEW,231,720.0
5,6,content_24abafed9707,3,STALE_AND_HIGH_DEMAND,PRIORITY_REVIEW,231,480.0
6,7,content_f4b3081037b3,3,STALE_AND_HIGH_DEMAND,PRIORITY_REVIEW,231,90.0
7,8,content_3f3576c295f5,2,STALE_ONLY,PRIORITY_REVIEW,373,0.0
8,9,content_55a5b1c46474,2,STALE_ONLY,PRIORITY_REVIEW,373,0.0
9,10,content_f6fdf87348f6,2,STALE_ONLY,PRIORITY_REVIEW,373,0.0


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [7]:
# Create a fresh Top-20 review table from the full ranked queue
top20_review = queue.head(20).copy()

def explain_rank(row):
    reasons = []

    if row["days_since_last_update"] >= 180:
        reasons.append("Content has not been updated for at least 180 days.")

    if pd.notna(row["search_volume"]):
        reasons.append(
            f"Search volume is {row['search_volume']:.0f}."
        )
    else:
        reasons.append("Search volume is missing, so demand is unknown.")

    return " ".join(reasons)

top20_review["confidence_note"] = top20_review.apply(
    lambda row: (
        "Medium: both staleness and search demand support review."
        if row["reason_code"] == "STALE_AND_HIGH_DEMAND"
        else "Low: only one priority signal supports review."
        if row["reason_code"] in ["STALE_ONLY", "HIGH_DEMAND_ONLY"]
        else "Low: neither priority signal is present."
    ),
    axis=1
)

top20_review["what_would_make_it_wrong"] = top20_review.apply(
    lambda row: (
        "The page may still be accurate, or its search demand may not justify a refresh."
        if row["reason_code"] == "STALE_AND_HIGH_DEMAND"
        else "The page may not need updating, or its search demand may be too low."
        if row["reason_code"] == "STALE_ONLY"
        else "Search demand alone may not justify a refresh."
        if row["reason_code"] == "HIGH_DEMAND_ONLY"
        else "The page could still have an opportunity that this simple score misses."
    ),
    axis=1
)

top20_review["reason"] = top20_review.apply(explain_rank, axis=1)

display(top20_review[[
    "rank",
    "content_id",
    "score",
    "action_label",
    "reason_code",
    "reason",
    "confidence_note",
    "what_would_make_it_wrong"
]])

,rank,content_id,score,action_label,reason_code,reason,confidence_note,what_would_make_it_wrong
0,1,content_02b0d6e30129,3,PRIORITY_REVIEW,STALE_AND_HIGH_DEMAND,Content has not been updated for at least 180 ...,Medium: both staleness and search demand suppo...,"The page may still be accurate, or its search ..."
1,2,content_7a888d3d99c8,3,PRIORITY_REVIEW,STALE_AND_HIGH_DEMAND,Content has not been updated for at least 180 ...,Medium: both staleness and search demand suppo...,"The page may still be accurate, or its search ..."
2,3,content_dd413158df3c,3,PRIORITY_REVIEW,STALE_AND_HIGH_DEMAND,Content has not been updated for at least 180 ...,Medium: both staleness and search demand suppo...,"The page may still be accurate, or its search ..."
3,4,content_bbca724138f2,3,PRIORITY_REVIEW,STALE_AND_HIGH_DEMAND,Content has not been updated for at least 180 ...,Medium: both staleness and search demand suppo...,"The page may still be accurate, or its search ..."
4,5,content_40e140ba2934,3,PRIORITY_REVIEW,STALE_AND_HIGH_DEMAND,Content has not been updated for at least 180 ...,Medium: both staleness and search demand suppo...,"The page may still be accurate, or its search ..."
5,6,content_24abafed9707,3,PRIORITY_REVIEW,STALE_AND_HIGH_DEMAND,Content has not been updated for at least 180 ...,Medium: both staleness and search demand suppo...,"The page may still be accurate, or its search ..."
6,7,content_f4b3081037b3,3,PRIORITY_REVIEW,STALE_AND_HIGH_DEMAND,Content has not been updated for at least 180 ...,Medium: both staleness and search demand suppo...,"The page may still be accurate, or its search ..."
7,8,content_3f3576c295f5,2,PRIORITY_REVIEW,STALE_ONLY,Content has not been updated for at least 180 ...,Low: only one priority signal supports review.,"The page may not need updating, or its search ..."
8,9,content_55a5b1c46474,2,PRIORITY_REVIEW,STALE_ONLY,Content has not been updated for at least 180 ...,Low: only one priority signal supports review.,"The page may not need updating, or its search ..."
9,10,content_f6fdf87348f6,2,PRIORITY_REVIEW,STALE_ONLY,Content has not been updated for at least 180 ...,Low: only one priority signal supports review.,"The page may not need updating, or its search ..."


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Show examples that deserve extra skepticism.
# These have only one of the two signals, rather than both.
weak_picks = queue[
    queue["reason_code"].isin(["STALE_ONLY", "HIGH_DEMAND_ONLY"])
].head(10).copy()

weak_picks["why_questionable"] = weak_picks["reason_code"].map({
    "STALE_ONLY": (
        "Age alone does not prove the page needs a refresh; "
        "search demand may be low or the content may still be accurate."
    ),
    "HIGH_DEMAND_ONLY": (
        "Search demand alone does not prove a refresh is needed; "
        "the page may already be current and useful."
    ),
})

print("Examples of weaker, single-signal picks:")
display(weak_picks)

# Leakage audit: these are the only columns used to calculate the score.
scoring_inputs = {"days_since_last_update", "search_volume"}

forbidden_inputs = {
    "trend_direction", "trend_pct",
    "impressions_90d", "clicks_90d",
    "pageviews_90d", "sessions_90d",
    "impressions_last_30d", "clicks_last_30d",
    "sessions_last_30d",
    "impressions_prev_30d", "clicks_prev_30d",
    "sessions_prev_30d",
    "ctr", "avg_position", "engagement_rate",
    "scroll_rate", "ai_traffic_pct",
    "freshness_tier", "impression_tier", "position_tier",
}

assert scoring_inputs.isdisjoint(forbidden_inputs)
assert "trend_direction" not in queue.columns
assert "trend_pct" not in queue.columns
assert len(queue) == len(df)
assert queue["rank"].is_unique


print("Leakage checks passed for the selected scoring inputs.")
print("Scoring inputs:", sorted(scoring_inputs))
print("No target label, target-derived trend, traffic-window, or existing tier flag was used in the score.")


Examples of weaker, single-signal picks:


,rank,content_id,score,reason_code,action_label,days_since_last_update,search_volume,why_questionable
7,8,content_3f3576c295f5,2,STALE_ONLY,PRIORITY_REVIEW,373,0.0,Age alone does not prove the page needs a refr...
8,9,content_55a5b1c46474,2,STALE_ONLY,PRIORITY_REVIEW,373,0.0,Age alone does not prove the page needs a refr...
9,10,content_f6fdf87348f6,2,STALE_ONLY,PRIORITY_REVIEW,373,0.0,Age alone does not prove the page needs a refr...
10,11,content_1b4ec72dafd4,2,STALE_ONLY,PRIORITY_REVIEW,372,0.0,Age alone does not prove the page needs a refr...
11,12,content_8d56efff1e71,2,STALE_ONLY,PRIORITY_REVIEW,372,0.0,Age alone does not prove the page needs a refr...
12,13,content_f01216059a6a,2,STALE_ONLY,PRIORITY_REVIEW,335,NaN,Age alone does not prove the page needs a refr...
13,14,content_06e19c6486b0,2,STALE_ONLY,PRIORITY_REVIEW,334,NaN,Age alone does not prove the page needs a refr...
14,15,content_e2b702f4f92b,2,STALE_ONLY,PRIORITY_REVIEW,334,NaN,Age alone does not prove the page needs a refr...
15,16,content_6476d1d8c050,2,STALE_ONLY,PRIORITY_REVIEW,313,10.0,Age alone does not prove the page needs a refr...
16,17,content_94991fe6268c,2,STALE_ONLY,PRIORITY_REVIEW,313,10.0,Age alone does not prove the page needs a refr...


Leakage checks passed for the selected scoring inputs.
Scoring inputs: ['days_since_last_update', 'search_volume']
No target label, target-derived trend, traffic-window, or existing tier flag was used in the score.


In [9]:
print("df columns:")
print(df.columns.tolist())

print("\nqueue columns:")
print(queue.columns.tolist())

print("\ntop20_review columns:")
print(top20_review.columns.tolist())

df columns:
['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']

queue columns:
['rank', 'content_id', 'score', 'reason_code', 'action_label', 'days_since_last_update', 'search_volume']

top20_review columns:
['rank', 'content_id', 'score', 'reason_code', 'action_label', 'days_sin

In [10]:
def explain_rank(row):
    parts = []

    if row["days_since_last_update"] >= 180:
        parts.append(
            f"stale: {int(row['days_since_last_update'])} days since update"
        )

    volume = row["search_volume"]
    if pd.notna(volume) and volume > 0 and volume >= high_demand_threshold:
        parts.append(
            f"search volume {volume:g} meets the high-demand threshold"
        )
    elif pd.isna(volume):
        parts.append("search volume is missing")

    if not parts:
        parts.append("neither selected signal triggered")

    return f"Score {int(row['score'])}/3; " + "; ".join(parts)


top20_review["why_it_ranked_here"] = top20_review.apply(
    explain_rank, axis=1
)

# Put the explanation next to the action and reason code
top20_review = top20_review[
    [
        "rank",
        "content_id",
        "action_label",
        "reason_code",
        "score",
        "why_it_ranked_here",
        "confidence_note",
        "what_would_make_it_wrong",
    ]
]

display(top20_review)

,rank,content_id,action_label,reason_code,score,why_it_ranked_here,confidence_note,what_would_make_it_wrong
0,1,content_02b0d6e30129,PRIORITY_REVIEW,STALE_AND_HIGH_DEMAND,3,Score 3/3; stale: 313 days since update; searc...,Medium: both staleness and search demand suppo...,"The page may still be accurate, or its search ..."
1,2,content_7a888d3d99c8,PRIORITY_REVIEW,STALE_AND_HIGH_DEMAND,3,Score 3/3; stale: 313 days since update; searc...,Medium: both staleness and search demand suppo...,"The page may still be accurate, or its search ..."
2,3,content_dd413158df3c,PRIORITY_REVIEW,STALE_AND_HIGH_DEMAND,3,Score 3/3; stale: 305 days since update; searc...,Medium: both staleness and search demand suppo...,"The page may still be accurate, or its search ..."
3,4,content_bbca724138f2,PRIORITY_REVIEW,STALE_AND_HIGH_DEMAND,3,Score 3/3; stale: 236 days since update; searc...,Medium: both staleness and search demand suppo...,"The page may still be accurate, or its search ..."
4,5,content_40e140ba2934,PRIORITY_REVIEW,STALE_AND_HIGH_DEMAND,3,Score 3/3; stale: 231 days since update; searc...,Medium: both staleness and search demand suppo...,"The page may still be accurate, or its search ..."
5,6,content_24abafed9707,PRIORITY_REVIEW,STALE_AND_HIGH_DEMAND,3,Score 3/3; stale: 231 days since update; searc...,Medium: both staleness and search demand suppo...,"The page may still be accurate, or its search ..."
6,7,content_f4b3081037b3,PRIORITY_REVIEW,STALE_AND_HIGH_DEMAND,3,Score 3/3; stale: 231 days since update; searc...,Medium: both staleness and search demand suppo...,"The page may still be accurate, or its search ..."
7,8,content_3f3576c295f5,PRIORITY_REVIEW,STALE_ONLY,2,Score 2/3; stale: 373 days since update,Low: only one priority signal supports review.,"The page may not need updating, or its search ..."
8,9,content_55a5b1c46474,PRIORITY_REVIEW,STALE_ONLY,2,Score 2/3; stale: 373 days since update,Low: only one priority signal supports review.,"The page may not need updating, or its search ..."
9,10,content_f6fdf87348f6,PRIORITY_REVIEW,STALE_ONLY,2,Score 2/3; stale: 373 days since update,Low: only one priority signal supports review.,"The page may not need updating, or its search ..."


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.